In [2]:
from captum.attr import configure_interpretable_embedding_layer, remove_interpretable_embedding_layer
from captum.attr import LayerIntegratedGradients, LayerGradientXActivation
from config import get_model_config
from models.interpretability import CaseTransformerInterpretability, CaptumWrapper
from models.transformer_helpers import encode_month_year_features, bond_one_hot, discretize_bond_amount
from models.transformer_model import CaseTransformerModel
from pathlib import Path
from transformers import AutoModel, AutoTokenizer, BertModel, BertTokenizer
from typing import Dict, List, Tuple, Optional, Any, Union
from utils.cache_directory_utils import CacheDirectoryManager
from training.teacher_forcing import build_teacher_forced_sequences
import csv
import gc
import matplotlib.pyplot as plt
import numpy as np
import pickle
import random
import seaborn as sns
import sys
import torch
import torch.nn as nn
import pandas as pd
import nltk
import string
from collections import defaultdict
from training.top_k_helper import predict_with_top_k_mlb

In [3]:
try:
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    from nltk.corpus import stopwords
    stop_words = set(stopwords.words('english'))

In [4]:
import os
import json
from pathlib import Path

parameters_filename = 'balanced_config_8.json'
PROJECT_ROOT = Path("/home/krosen5/Court Case Sequence Transformer Model/court_case_transformer_project")
parameters_path = PROJECT_ROOT / "parameters_jsons" / parameters_filename
with open(parameters_path, "r", encoding="utf-8") as file:
    parameters_dict = json.load(file)

In [5]:
def load_model_for_captum(
    model_cache_manager: CacheDirectoryManager,
    model_name="final_model",
    device=None,
    verbose=True
):
    model_dir = model_cache_manager.get_cache_dir('exported_models')
    
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print("="*70)
    print("LOADING MODEL FOR CAPTUM INTERPRETABILITY")
    print("="*70)
    print(f"Model directory: {model_dir}")
    print(f"Device: {device}")
    print(f"Verbose mode: {verbose}")
    print()
    
    # 1. Load model configuration
    config_path = model_dir / 'model_config.pkl'
    if not config_path.exists():
        raise FileNotFoundError(f"Model config not found: {config_path}")
    
    with open(config_path, 'rb') as f:
        config_dict = pickle.load(f)
    print(f"✅ Loaded model configuration")
    
    # 2. Update global config
    model_config = get_model_config()
    if config_dict['min_year'] != model_config.min_year or config_dict['max_year'] != model_config.max_year:
        model_config.update_date_range(config_dict['min_year'], config_dict['max_year'])
    
    # 3. Load vocabularies
    charge_vocab_path = model_dir / 'charge_vocab.pkl'
    with open(charge_vocab_path, 'rb') as f:
        charge_vocab = pickle.load(f)
    print(f"✅ Loaded charge vocabulary (size: {charge_vocab.size})")
    
    mag_vocab_path = model_dir / 'mag_vocab.pkl'
    with open(mag_vocab_path, 'rb') as f:
        mag_vocab = pickle.load(f)
    print(f"✅ Loaded magistrate vocabulary (size: {mag_vocab.size})")
    
    # 4. Load label processor
    label_processor_path = model_dir / 'label_processor.pkl'
    with open(label_processor_path, 'rb') as f:
        label_processor = pickle.load(f)
    print(f"✅ Loaded label processor ({len(label_processor.label_names)} labels)")
    
    # 5. Check BERT cache
    bert_cache_dir = model_cache_manager.get_cache_dir('bert')
    if not bert_cache_dir.exists():
        raise FileNotFoundError(
            f"BERT cache directory not found: {bert_cache_dir}\n"
            "Captum analysis requires BERT embeddings to be available."
        )
    print(f"✅ Found BERT cache: {bert_cache_dir}")
    
    # 6. Create model with interpretability enabled
    print("\n🔧 Creating model architecture...")
    model = CaseTransformerModel(
        charge_vocab=charge_vocab,
        mag_vocab=mag_vocab,
        bert_cache_dir=bert_cache_dir,
        num_labels=config_dict['num_labels'],
        hidden_dim=config_dict['hidden_dim'],
        max_len=config_dict['max_len'],
        bond_dim=config_dict['bond_dim'],
        temporal_dim=config_dict['temporal_dim'],
        enable_interpretability=True  # IMPORTANT: Enable for Captum,
    )
    
    # 7. Load weights
    model_weights_path = model_dir / f"{model_name}.pt"
    if not model_weights_path.exists():
        raise FileNotFoundError(f"Model weights not found: {model_weights_path}")
    
    state_dict = torch.load(model_weights_path, map_location=device)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    print(f"✅ Loaded model weights")
    
    # 8. Disable cuDNN for reproducibility (required for Captum)
    if torch.cuda.is_available():
        original_cudnn_state = torch.backends.cudnn.enabled
        torch.backends.cudnn.enabled = False
        print(f"✅ Disabled cuDNN for reproducibility")
    
    # 9. Create interpretability wrapper
    print("\n🔬 Initializing Captum interpretability wrapper...")
    interpreter = CaseTransformerInterpretability(
        model=model,
        label_processor=label_processor,
        verbose=verbose,
        device=device
    )
    print(f"✅ Interpretability wrapper ready")
    
    print("\n" + "="*70)
    print("SETUP COMPLETE - READY FOR CAPTUM ANALYSIS")
    print("="*70)
    print(f"\nAvailable attribution methods:")
    print("  - IntegratedGradients (recommended)")
    print("  - DeepLift")
    print("  - InputXGradient")
    print("  - Saliency")
    print(f"\nLabel names: {label_processor.label_names[:3]}... (total: {len(label_processor.label_names)})")
    print("="*70 + "\n")
    
    return {
        'model': model,
        'label_processor': label_processor,
        'interpreter': interpreter,
        'device': device,
        'config': config_dict,
        'charge_vocab': charge_vocab,
        'mag_vocab': mag_vocab
    }

In [6]:
folder = rf'/data/krosen5/cache/transformers/{parameters_dict["folder_name_directory"]}/'
if not os.path.exists(folder):
    os.makedirs(folder)
model_cache_manager = CacheDirectoryManager(base_path=folder)

components = load_model_for_captum(model_cache_manager, model_name="final_model_mlb_best_fold_1", verbose=False)
model = components['model']
label_processor = components['label_processor']
device = components['device']

LOADING MODEL FOR CAPTUM INTERPRETABILITY
Model directory: /data/krosen5/cache/transformers/balanced_8/exported_models
Device: cuda
Verbose mode: False

✅ Loaded model configuration
Date range updated: 2008 to 2025
✅ Loaded charge vocabulary (size: 2139)
✅ Loaded magistrate vocabulary (size: 6)
✅ Loaded label processor (20 labels)
✅ Found BERT cache: /data/krosen5/cache/transformers/balanced_8/bert_streaming

🔧 Creating model architecture...
✅ Loaded model weights
✅ Disabled cuDNN for reproducibility

🔬 Initializing Captum interpretability wrapper...
✅ Interpretability wrapper ready

SETUP COMPLETE - READY FOR CAPTUM ANALYSIS

Available attribution methods:
  - IntegratedGradients (recommended)
  - DeepLift
  - InputXGradient
  - Saliency

Label names: ['FIRST_APPEARANCE_HEARING_OCCURRED', 'BOND_SET', 'BOND_NOT_SET']... (total: 20)



In [7]:
def list_labels(config: Dict) -> List[Tuple[int, str]]:
    label_names = config.get('label_names', [])
    return list(enumerate(label_names))

for idx, name in list_labels(components['config']):
    print(f"{idx}: {name}")

0: FIRST_APPEARANCE_HEARING_OCCURRED
1: BOND_SET
2: BOND_NOT_SET
3: DEFENDANT_RELEASED_ON_OWN_RECOGNIZANCE
4: THE_COURT_APPOINTED_THE_ORLEANS_PUBLIC_DEFENDER
5: ANKLE_MONITORING_IMPOSED
6: DOMESTIC_STAY_AWAY_ORDER_IMPOSED
7: CHARGES_ACCEPTED
8: CHARGES_REFUSED
9: PROBABLE_CAUSE_FOUND
10: PROBABLE_CAUSE_NOT_FOUND
11: DEFENDANT_IN_CUSTODY_AND_NOT_IN_COURT
12: THE_STATE_WAS_NOT_PRESENT
13: PRE_TRIAL_CONFERENCE_HELD
14: CONTINUANCE_OCCURRED
15: PLED_GUILTY_TO_CHARGES
16: PLED_NOT_GUILTY_TO_CHARGES
17: TRIAL_VERDICT_REACHED
18: SENTENCE_THAT_INCLUDES_JAIL_TIME_WAS_IMPOSED
19: SENTENCE_THAT_DOES_NOT_INCLUDE_JAIL_TIME_WAS_IMPOSED


In [8]:
class BertToLabelWrapper(nn.Module):
    """
    Wrapper that connects BERT token embeddings directly to label predictions.
    Strictly aligns with CaseTransformerModel feature ordering for interpretability.
    """
    
    def __init__(
        self,
        bert_model: nn.Module,
        case_transformer: nn.Module,
        target_proceeding_idx: int = -1
    ):
        super().__init__()
        self.bert_model = bert_model
        self.case_transformer = case_transformer
        self.target_proceeding_idx = target_proceeding_idx
        
        # 1. Capture dynamic dimensions
        self.hidden_dim = case_transformer.hidden_dim
        
        # FIX: Hardcode to 768 (Standard BERT) or try to get it from the model
        # The error occurred because case_transformer.text_emb doesn't have .out_features
        if hasattr(case_transformer.text_emb, 'out_features'):
            self.bert_out_dim = case_transformer.text_emb.out_features
        elif hasattr(case_transformer.text_emb, 'embedding_dim'):
             self.bert_out_dim = case_transformer.text_emb.embedding_dim
        else:
            # Fallback for bert-base-uncased
            self.bert_out_dim = 768 
        
        # 2. Calculate slicing indices based on 'encode_proceeding' order:
        # Order: [Def_Emb, Case_Emb, TEXT_EMB, Mag_Emb, Prior_Lbl, Prior_Chg, Features, Temporal]
        
        # Def_Emb (hidden//4) + Case_Emb (hidden//4)
        self.pre_text_dim = (self.hidden_dim // 4) + (self.hidden_dim // 4)
        
    def forward(
        self,
        inputs_embeds: torch.Tensor,            # (batch, seq_len, 768) <- From Captum
        other_features_concat: torch.Tensor,    # (batch, total_unfused_dim) <- Static features
        attention_mask: torch.Tensor = None,
    ) -> torch.Tensor:
        """
        Forward pass that:
        1. Contextualizes BERT embeddings
        2. Swaps the static text embedding in 'other_features' with the new BERT output
        3. Runs the full CaseTransformer prediction head
        """
        # --- Step 1: Get contextualized CLS token from BERT ---
        bert_outputs = self.bert_model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask
        )
        
        # Extract CLS token (index 0)
        # Shape: (batch, bert_out_dim)
        text_embedding = bert_outputs.last_hidden_state[:, 0, :]
        
        # --- Step 2: Surgical replacement of text features ---
        # Slice A: Everything before text
        pre_text_features = other_features_concat[:, :self.pre_text_dim]
        
        # Slice B: Everything after text
        # Skip the slice corresponding to the *old* cached text embedding
        post_text_start = self.pre_text_dim + self.bert_out_dim
        post_text_features = other_features_concat[:, post_text_start:]
        
        # Reassemble: [Pre, NEW_TEXT, Post]
        combined_features = torch.cat(
            [pre_text_features, text_embedding, post_text_features], 
            dim=1
        )
        
        # --- Step 3: CaseTransformer Logic ---
        x = self.case_transformer.fuse(combined_features)
        x = x.unsqueeze(1) # Batch, Seq_Len=1, Hidden
        
        # Add Positional Encoding
        seq_idx = self.target_proceeding_idx
        if seq_idx < 0: seq_idx = 0 
        
        # Clamp to max position
        max_pos = self.case_transformer.pos.size(1)
        if seq_idx >= max_pos: seq_idx = max_pos - 1
            
        pos_vector = self.case_transformer.pos[:, seq_idx : seq_idx+1, :]
        x = x + pos_vector

        # Encoder & Classifier
        encoder_output = self.case_transformer.encoder(x)
        logits = self.case_transformer.classifier(encoder_output[:, -1, :])
        
        return logits


In [9]:

class BertTokenAttributionAnalyzer:
    """
    Per-word BERT attribution analysis for CaseTransformerModel.
    Connects raw text -> BERT -> Fusion -> Classifier.
    """
    
    def __init__(
        self,
        model: nn.Module,
        label_processor: Any,
        config: Dict,
        bert_model_name: str = "bert-base-uncased",
        device: Optional[torch.device] = None,
        verbose: bool = True
    ):
        self.model = model
        self.model.eval()
        self.label_processor = label_processor
        self.config = config
        self.label_names = config.get('label_names', getattr(label_processor, 'label_names', []))
        self.num_labels = len(self.label_names)
        self.verbose = verbose
        
        if device is None:
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.device = device
        self.model.to(device)
        
        # Initialize BERT for attribution analysis
        if self.verbose:
            print(f"🔤 Loading BERT tokenizer and model: {bert_model_name}")
        self.bert_tokenizer = BertTokenizer.from_pretrained(bert_model_name)
        self.bert_model = BertModel.from_pretrained(bert_model_name)
        self.bert_model.to(device)
        self.bert_model.eval()
        
        # Special token IDs
        self.ref_token_id = self.bert_tokenizer.pad_token_id
        self.sep_token_id = self.bert_tokenizer.sep_token_id
        self.cls_token_id = self.bert_tokenizer.cls_token_id
        
        # Initialize the wrapper (Wrapper connects BERT -> CaseTransformer)
        self.wrapper = BertToLabelWrapper(self.bert_model, self.model)
        self.wrapper.to(device)
        
        if self.verbose:
            print(f"✅ BertTokenAttributionAnalyzer initialized on {device}")

    # -------------------------------------------------------------------------
    # Input Construction
    # -------------------------------------------------------------------------
    
    def construct_input_ref_pair(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        text_ids = self.bert_tokenizer.encode(text, add_special_tokens=False, max_length=510, truncation=True)
        input_ids = [self.cls_token_id] + text_ids + [self.sep_token_id]
        ref_input_ids = [self.cls_token_id] + [self.ref_token_id] * len(text_ids) + [self.sep_token_id]
        
        return (
            torch.tensor([input_ids], device=self.device),
            torch.tensor([ref_input_ids], device=self.device)
        )
    
    def get_bert_embeddings(self, input_ids):
        return self.bert_model.embeddings(input_ids)

    # -------------------------------------------------------------------------
    # Core Attribution Method
    # -------------------------------------------------------------------------
    
    def attribute_tokens_for_sequence(
        self,
        seq: List[Dict],
        target_label_idx: int,
        target_proceeding_idx: int = -1,
        n_steps: int = 50
    ) -> Dict[str, Any]:
        """
        Compute attributions for a specific proceeding in a sequence.
        Crucially: Integrates static features (Defendant, Bond, etc.)
        """
        # Handle index
        if target_proceeding_idx < 0:
            target_proceeding_idx = len(seq) + target_proceeding_idx
            
        proceeding = seq[target_proceeding_idx]
        text = proceeding.get('text', '') or proceeding.get('description', '')
        
        if not text.strip():
            raise ValueError(f"Proceeding at index {target_proceeding_idx} has no text")
            
        # 1. Update Wrapper State
        self.wrapper.target_proceeding_idx = target_proceeding_idx
        
        # 2. Extract Static Features using the Model's logic
        # This calls the method you added to CaseTransformerModel
        static_features = self.model.encode_proceeding_unfused(proceeding, target_proceeding_idx)
        static_features = static_features.to(self.device)

        # 3. Prepare BERT Inputs
        input_ids, ref_input_ids = self.construct_input_ref_pair(text)
        attention_mask = torch.ones_like(input_ids)
        
        # Get embeddings
        input_embeddings = self.get_bert_embeddings(input_ids)
        ref_embeddings = self.get_bert_embeddings(ref_input_ids)
        
        # 4. Initialize LIG
        # We attribute on the BERT embeddings, passing through our wrapper
        lig = LayerIntegratedGradients(self.wrapper, self.bert_model.embeddings)
        
        # 5. Compute Attributions
        # We pass static_features and attention_mask as additional args to forward()
        attributions, delta = lig.attribute(
            inputs=input_embeddings,
            baselines=ref_embeddings,
            target=target_label_idx,
            additional_forward_args=(static_features, attention_mask),
            n_steps=n_steps,
            return_convergence_delta=True
        )
        
        # 6. Process Results
        indices = input_ids[0].detach().tolist()
        tokens = self.bert_tokenizer.convert_ids_to_tokens(indices)
        
        # Summarize (Sum across embedding dim 768)
        attributions_sum = attributions.sum(dim=-1).squeeze(0)
        norm = torch.norm(attributions_sum)
        if norm > 0:
            attributions_sum = attributions_sum / norm
            
        return {
            'tokens': tokens,
            'attributions_sum': attributions_sum.detach().cpu().numpy(),
            'target_label_idx': target_label_idx,
            'target_label': self.label_names[target_label_idx] if self.label_names else str(target_label_idx),
            'convergence_delta': delta.item(),
            'text': text,
            'proceeding_idx': target_proceeding_idx
        }
    def _merge_compound_tokens(self, words, scores):
        """
        Second-pass merge for dates, times, and currency amounts
        that BERT splits into separate tokens.
        e.g. '$', '8', ',', '000' -> '$8,000'
            '4', '/', '10', '/', '2024' -> '4/10/2024'
            '3', ':', '00', 'pm' -> '3:00pm'
        """
        merged_words = []
        merged_scores = []
        i = 0

        while i < len(words):
            w = words[i]
            s = float(scores[i])

            # Currency: $ followed by digits
            if w == '$' and i + 1 < len(words) and words[i + 1][0].isdigit():
                j = i + 1
                while j < len(words) and (words[j][0].isdigit() or words[j] in [',', '.']):
                    w += words[j]
                    s += float(scores[j])
                    j += 1
                i = j

            # Date (digit/digit/digit) or time (digit:digit[am/pm])
            elif w[0].isdigit() and i + 1 < len(words) and words[i + 1] in ['/', ':']:
                separator = words[i + 1]
                j = i + 1
                while j < len(words):
                    nw = words[j]
                    if nw in ['/', ':']:
                        w += nw
                        s += float(scores[j])
                        j += 1
                    elif nw[0].isdigit():
                        w += nw
                        s += float(scores[j])
                        j += 1
                    elif nw.lower() in ['am', 'pm'] and separator == ':':
                        w += nw
                        s += float(scores[j])
                        j += 1
                        break
                    else:
                        break
                i = j

            else:
                i += 1

            merged_words.append(w)
            merged_scores.append(s)

        return merged_words, merged_scores
    def aggregate_word_attributions(
        self,
        results: Dict[str, Any],
        remove_stopwords: bool = True,
        extra_stopwords: Optional[set] = None
    ) -> Dict[str, float]:
        """
        Reassemble BERT subword tokens into full words and aggregate attribution scores.
        Only filters [CLS], [SEP], [PAD] — keeps numbers, punctuation, alphanumeric codes.

        Args:
            results:          Dict returned from attribute_tokens_for_sequence()
            remove_stopwords: If True, filter NLTK English stopwords
            stopword_lang:    Language for NLTK stopwords (default: 'english')
            extra_stopwords:  Optional set of additional words to filter (e.g. legal boilerplate)

        Returns:
            Dict of {word: aggregated_score}, sorted by |score| descending
        """
        tokens = results['tokens']
        attrs  = results['attributions_sum']

        # --- Step 1: Reassemble ## subword pieces into full words ---
        words        = []
        word_scores  = []
        current_word  = None
        current_score = 0.0

        SKIP = {'[CLS]', '[SEP]', '[PAD]'}

        for token, score in zip(tokens, attrs):
            if token in SKIP:
                continue

            if token.startswith('##'):
                if current_word is not None:
                    current_word  += token[2:]
                    current_score += float(score)
            else:
                if current_word is not None:
                    words.append(current_word)
                    word_scores.append(current_score)
                current_word  = token
                current_score = float(score)

        # Flush the final word
        if current_word is not None:
            words.append(current_word)
            word_scores.append(current_score)

        # --- Step 2: Merge dates, times, currency ---
        words, word_scores = self._merge_compound_tokens(words, word_scores)

        # --- Step 3: Build stopword set ---
        stop_words = set()
        if remove_stopwords:
            stop_words = set(stopwords.words('english'))

        if extra_stopwords:
            stop_words |= {w.lower() for w in extra_stopwords}

        # --- Step 3: Aggregate, then filter stopwords ---
        word_importances = {}
        word_counts = {}
        for word, score in zip(words, word_scores):
            key = word.strip(string.punctuation).lower()
            if not key or key in stop_words:
                continue
            count = word_counts.get(key, 0)
            # First occurrence keeps plain name, subsequent ones get _2, _3 etc.
            entry_key = key if count == 0 else f"{key}_{count + 1}"
            word_importances[entry_key] = score
            word_counts[key] = count + 1

        return word_importances


    # -------------------------------------------------------------------------
    # Visualization (Standard)
    # -------------------------------------------------------------------------

    def visualize_token_attributions(self, results: Dict[str, Any], save_path: str = None):
        tokens = results['tokens']
        attrs = results['attributions_sum']
        label = results['target_label']
        
        # Filter out CLS/SEP for cleaner plot
        filtered_tokens = [t for t in tokens if t not in ['[CLS]', '[SEP]', '[PAD]']]
        filtered_attrs  = [a for t, a in zip(tokens, attrs) if t not in ['[CLS]', '[SEP]', '[PAD]']]


        fig, ax = plt.subplots(figsize=(15, 5))
        colors = ['#2ca02c' if a > 0 else '#d62728' for a in filtered_attrs] # Green/Red
        
        ax.bar(range(len(filtered_tokens)), filtered_attrs, color=colors)
        ax.set_xticks(range(len(filtered_tokens)))
        ax.set_xticklabels(filtered_tokens, rotation=90, ha='right', fontsize=9)
        ax.axhline(0, color='black', linewidth=0.5)
        ax.set_title(f"Token Contributions to Label: {label}")
        ax.set_ylabel("Attribution Score")
        
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path)
            print(f"Saved plot to {save_path}")
        plt.show()
    
    def visualize_word_attributions(
        self,
        word_importances: Dict[str, float],
        save_path: str = None
    ):
        items  = list(word_importances.items())
        words  = [item[0] for item in items]
        scores = [item[1] for item in items]

        fig, ax = plt.subplots(figsize=(15, 5))
        colors = ['#2ca02c' if s > 0 else '#d62728' for s in scores]

        ax.bar(range(len(words)), scores, color=colors)
        ax.set_xticks(range(len(words)))
        ax.set_xticklabels(words, rotation=90, ha='right', fontsize=9)
        ax.axhline(0, color='black', linewidth=0.5)
        ax.set_title(f"Word Contributions to Label")
        ax.set_ylabel("Attribution Score")

        plt.tight_layout()
        if save_path:
            plt.savefig(save_path)
            print(f"Saved plot to {save_path}")
        plt.show()

In [10]:
def explain_proceeding(
    model, 
    seq: List[Dict], 
    target_idx: int, 
    label_idx: int, 
    config: Dict
):
    """
    Helper to quickly run the analyzer on a sequence
    """
    analyzer = BertTokenAttributionAnalyzer(
        model=model, 
        label_processor=None, # Optional if labels in config
        config=config
    )
    
    results = analyzer.attribute_tokens_for_sequence(
        seq=seq, 
        target_proceeding_idx=target_idx, 
        target_label_idx=label_idx
    )
    
    analyzer.visualize_token_attributions(results)
    return results

In [11]:
TARGET_LABELS = [
    "PLED_GUILTY_TO_CHARGES",
    "PLED_NOT_GUILTY_TO_CHARGES",
    "TRIAL_VERDICT_REACHED",
    "SENTENCE_THAT_INCLUDES_JAIL_TIME_WAS_IMPOSED",
    "SENTENCE_THAT_DOES_NOT_INCLUDE_JAIL_TIME_WAS_IMPOSED",
    "CONTINUANCE_OCCURRED"
]

MAX_INSTANCES_PER_LABEL = None
MIN_WORDS = 5
NUMBER_OF_INTEGRATED_GRADIENTS_STEPS = 50
TOP_N_WORDS_TO_DISPLAY = 30

In [12]:
import pickle
import torch

sequences_path = model_cache_manager.get_cache_dir('datasets') / 'sequences_and_labels.pkl'
if not sequences_path.exists():
    print(f"❌ Error: File not found at {sequences_path}")

with open(sequences_path, 'rb') as f:
    sequences, labels = pickle.load(f)

print(f"✅ Loaded {len(sequences)} sequences.")

✅ Loaded 2533 sequences.


In [13]:
import random
def find_all_instances_of_label(sequences, labels, target_labels, max_per_label=None, seed=42):
    instances = defaultdict(list)
    for label_name in target_labels:
        for sequence_index, label_seq in enumerate(labels):
            for label_step_index, step_labels in enumerate(label_seq):
                if label_name in step_labels and label_step_index >= 1:
                    if label_step_index - 1 < len(sequences[sequence_index]):
                        instances[label_name].append((sequence_index, label_step_index - 1))
        if max_per_label is not None and len(instances[label_name]) > max_per_label:
            rng = random.Random(seed)
            instances[label_name] = rng.sample(instances[label_name], max_per_label)
    return dict(instances)

In [14]:
all_instances = find_all_instances_of_label(
    sequences=sequences,
    labels=labels,
    target_labels=TARGET_LABELS,
    max_per_label=MAX_INSTANCES_PER_LABEL,
)

print("Instances found per label:")
for label, inst_list in all_instances.items():
    print(f"  {label}: {len(inst_list)} instances")

Instances found per label:
  PLED_GUILTY_TO_CHARGES: 1077 instances
  PLED_NOT_GUILTY_TO_CHARGES: 1633 instances
  TRIAL_VERDICT_REACHED: 404 instances
  SENTENCE_THAT_INCLUDES_JAIL_TIME_WAS_IMPOSED: 806 instances
  SENTENCE_THAT_DOES_NOT_INCLUDE_JAIL_TIME_WAS_IMPOSED: 401 instances
  CONTINUANCE_OCCURRED: 5370 instances


In [15]:
def build_teacher_seq_for_instance(sequences, labels, si, li, label_processor):
    seq_slice    = sequences[si][:li + 1]   # length li+1; seq[li] never touched
    labels_slice = labels[si][:li + 1]      # labels[0...li]

    labels_binary = [
        lbl.cpu().numpy() if isinstance(lbl, torch.Tensor) else np.array(lbl, dtype=float)
        for lbl in label_processor.transform_sequence(labels_slice)
    ]

    teacher_seq, _ = build_teacher_forced_sequences(
        seq_slice,
        labels_binary,
        label_processor,
        include_initial_state=True
    )
    return teacher_seq

In [16]:
import csv
label_names = label_processor.label_names

OUTPUT_ROOT_FOLDER = Path(model_cache_manager.get_cache_dir('interpretability')) / 'captum_attributions'
OUTPUT_ROOT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Output root folder: {OUTPUT_ROOT_FOLDER}")

analyzer = BertTokenAttributionAnalyzer(model = model, label_processor=label_processor, config={}, device=device, verbose=False)

label_word_scores = defaultdict(lambda: defaultdict(list))

Output root folder: /data/krosen5/cache/transformers/balanced_8/interpretability/captum_attributions


In [17]:
def save_token_chart(results, save_path):
    tokens = results['tokens']
    attrs = results['attributions_sum']
    label = results['target_label']
    
    filtered_tokens = [t for t in tokens if t not in ['[CLS]', '[SEP]', '[PAD]']]
    filtered_attrs  = [a for t, a in zip(tokens, attrs) if t not in ['[CLS]', '[SEP]', '[PAD]']]

    fig, ax = plt.subplots(figsize=(max(15, len(filtered_tokens) * 0.25), 5))
    colors = ['#2ca02c' if a > 0 else '#d62728' for a in filtered_attrs]
    
    ax.bar(range(len(filtered_tokens)), filtered_attrs, color=colors)
    ax.set_xticks(range(len(filtered_tokens)))
    ax.set_xticklabels(filtered_tokens, rotation=90, ha='right', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f"Token Attributions — {label}\n"
                 f"seq={results.get('seq_idx','?')}  step={results.get('label_step','?')}  "
                 f"proc_idx={results['proceeding_idx']}", fontsize=9)
    ax.set_ylabel("Attribution Score")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def save_word_chart(word_importances, label_name, seq_index, label_step, save_path):
    items = list(word_importances.items())
    words = [w for w, _ in items]
    scores = [s for _, s in items]
    
    fig, ax = plt.subplots(figsize=(max(15, len(words) * 0.3), 5))
    colors  = ['#2ca02c' if s > 0 else '#d62728' for s in scores]
    ax.bar(range(len(words)), scores, color=colors)
    ax.set_xticks(range(len(words)))
    ax.set_xticklabels(words, rotation=90, ha='right', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f"Word Attributions — {label_name}\n"
                 f"seq={seq_index}  step={label_step}", fontsize=9)
    ax.set_ylabel("Attribution Score")
    plt.tight_layout()
    plt.savefig(save_path, dpi=120)
    plt.close(fig)

def save_scores_csv(results, word_importances, save_path):
    tokens = results['tokens']
    attributions = results['attributions_sum']
    word_rows = list(word_importances.items())
    token_rows = [(t, float(a)) for t, a in zip(tokens, attributions) if t not in ['[CLS]', '[SEP]', '[PAD]']]
    max_len = max(len(token_rows), len(word_rows))
    with open(save_path, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['Token', 'Token Attribution', 'Word', 'Word Attribution'])
        for i in range(max_len):
            token, token_score = token_rows[i] if i < len(token_rows) else ('', '')
            word, word_score = word_rows[i] if i < len(word_rows) else ('', '')
            writer.writerow([token, token_score, word, word_score])

In [ ]:
label_word_scores = defaultdict(lambda: defaultdict(list))

for label_name, instances in all_instances.items():
    label_idx    = label_names.index(label_name)
    n            = len(instances)
    label_folder = OUTPUT_ROOT_FOLDER / label_name
    label_folder.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Processing: {label_name}  ({n} instances)")
    print(f"Saving to:  {label_folder}")
    print(f"{'='*60}")

    skipped = 0
    not_predicted = 0
    for i, (si, li) in enumerate(instances):

        # Check text before building teacher seq (cheap early exit)
        raw_proc = sequences[si][li - 1]
        text = raw_proc.get('text', '') or raw_proc.get('description', '')
        if not text or len(text.split()) < MIN_WORDS:
            skipped += 1
            continue

        prefix = label_folder / f"instance_{i:03d}_seq{si}_step{li:02d}"

        try:
            # Build teacher-forced sequence for this instance
            # teacher_seq[li] = sequences[si][li-1] with prior_labels = labels[li-1]
            teacher_seq = build_teacher_seq_for_instance(
                sequences, labels, si, li, label_processor
            )
            target_proc_idx = li  # position in teacher_seq

            k_labels = len(labels[si][li])
            if k_labels == 0:
                skipped += 1
                continue
            
            with torch.no_grad():
                logits = model(teacher_seq, return_all_timesteps=True)
                probs = torch.sigmoid(logits).squeeze(0).cpu()
            
            probability_at_label_step_index = probs[li]
            pred_binary = predict_with_top_k_mlb(probability_at_label_step_index, k_labels, label_processor)

            if pred_binary[label_idx] == 0:
                not_predicted += 1
                continue

            del logits, probs, probability_at_label_step_index, pred_binary
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            

            results = analyzer.attribute_tokens_for_sequence(
                seq=teacher_seq,
                target_label_idx=label_idx,
                target_proceeding_idx=target_proc_idx,
                n_steps=NUMBER_OF_INTEGRATED_GRADIENTS_STEPS
            )
            results['seq_idx']    = si
            results['label_step'] = li

            word_imps = analyzer.aggregate_word_attributions(results, remove_stopwords=True)

            # Accumulate for aggregate
            for word, score in word_imps.items():
                label_word_scores[label_name][word].append(score)

            # Save per-instance files
            save_token_chart(results, str(prefix) + '_tokens.png')
            save_word_chart(word_imps, label_name, si, li, str(prefix) + '_words.png')
            save_scores_csv(results,  word_imps,   str(prefix) + '_scores.csv')

        except Exception as e:
            skipped += 1
            if 'memory' in str(e).lower():
                print(f"  ⚠️  OOM at ({si}, {li}) — clearing cache")
            else:
                print(f"  ⚠️  Skipping ({si}, {li}): {e}")

        if (i + 1) % 10 == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"  [{i+1}/{n}] done  (skipped so far: {skipped})")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    n_saved = n - skipped
    print(f"  ✅ {label_name}: {n_saved} saved, {skipped} skipped")
    print(f"     Files: {n_saved * 3} (tokens.png + words.png + scores.csv per instance)")


Processing: PLED_GUILTY_TO_CHARGES  (1077 instances)
Saving to:  /data/krosen5/cache/transformers/balanced_8/interpretability/captum_attributions/PLED_GUILTY_TO_CHARGES
  [90/1077] done  (skipped so far: 0)
  [140/1077] done  (skipped so far: 0)
  [180/1077] done  (skipped so far: 0)
  [200/1077] done  (skipped so far: 0)
  [340/1077] done  (skipped so far: 0)
  [820/1077] done  (skipped so far: 1)
  [830/1077] done  (skipped so far: 1)
  ✅ PLED_GUILTY_TO_CHARGES: 1076 saved, 1 skipped
     Files: 3228 (tokens.png + words.png + scores.csv per instance)

Processing: PLED_NOT_GUILTY_TO_CHARGES  (1633 instances)
Saving to:  /data/krosen5/cache/transformers/balanced_8/interpretability/captum_attributions/PLED_NOT_GUILTY_TO_CHARGES
  [20/1633] done  (skipped so far: 0)
  [160/1633] done  (skipped so far: 0)
  [180/1633] done  (skipped so far: 0)
  [240/1633] done  (skipped so far: 0)
  [250/1633] done  (skipped so far: 0)
  [260/1633] done  (skipped so far: 0)
  [400/1633] done  (skipped s

In [ ]:
label_top_words = {}

for label_name, word_score_lists in label_word_scores.items():
    mean_scores  = {word: np.mean(scores) for word, scores in word_score_lists.items()}
    sorted_words = sorted(mean_scores.items(), key=lambda x: abs(x[1]), reverse=True)
    label_top_words[label_name] = sorted_words

    n_instances = len(all_instances.get(label_name, []))
    print(f"\n{'='*60}")
    print(f"🔤 Top 20 words for '{label_name}'  (n={n_instances}):")
    print(f"{'='*60}")
    for word, score in sorted_words[:20]:
        bar  = '█' * int(abs(score) * 50)
        sign = '+' if score > 0 else '-'
        print(f"  {sign}{abs(score):.4f}  {bar}  {word}")

In [ ]:
for label_name, sorted_words in label_top_words.items():
    words  = [w for w, _ in sorted_words[:TOP_N_WORDS_TO_DISPLAY]]
    scores = [s for _, s in sorted_words[:TOP_N_WORDS_TO_DISPLAY]]
    n_inst = len(all_instances.get(label_name, []))

    fig, ax = plt.subplots(figsize=(16, 5))
    colors  = ['#2ca02c' if s > 0 else '#d62728' for s in scores]
    ax.bar(range(len(words)), scores, color=colors)
    ax.set_xticks(range(len(words)))
    ax.set_xticklabels(words, rotation=90, ha='right', fontsize=9)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f"Mean Word Attributions — {label_name}  (n={n_inst})", fontsize=11)
    ax.set_ylabel("Mean Attribution Score")
    plt.tight_layout()

    agg_path = OUTPUT_ROOT_FOLDER / label_name / f"_AGGREGATE_top{TOP_N_WORDS_TO_DISPLAY}_words.png"
    plt.savefig(agg_path, dpi=150)
    plt.show()
    print(f"Saved: {agg_path}")

In [ ]:
output_path = OUTPUT_ROOT_FOLDER / 'captum_word_attributions.pkl'

save_payload = {
    'label_top_words':   label_top_words,
    'label_word_scores': {k: dict(v) for k, v in label_word_scores.items()},
    'all_instances':     all_instances,
    'target_labels':     TARGET_LABELS,
    'output_root':       str(OUTPUT_ROOT_FOLDER),
    'config': {
        'max_instances_per_label': MAX_INSTANCES_PER_LABEL,
        'min_words':               MIN_WORDS,
        'number_of_integrated_gradients_steps': NUMBER_OF_INTEGRATED_GRADIENTS_STEPS,
        'top_n_words_to_display': TOP_N_WORDS_TO_DISPLAY,
    }
}

with open(output_path, 'wb') as f:
    pickle.dump(save_payload, f)

print(f"✅ Saved to {output_path}")
print()
print("Output folder structure:")
for label_name in TARGET_LABELS:
    label_folder = OUTPUT_ROOT_FOLDER / label_name
    if label_folder.exists():
        files = list(label_folder.iterdir())
        print(f"  {label_name}/  ({len(files)} files)")